# LayerNorm from Scratch

### Check version

In [1]:
from importlib.metadata import version
print("torch version: ", version("torch"))

torch version:  2.7.0+cu128


### Implementation

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

import math

In [3]:
class LayerNorm(nn.Module):
    """
    formula:
    output = ((x - mean(x)) / (std(x) + eps)) * gamma + beta
    gamma and beta are learned parameters
    """
    def __init__(self, D, eps=1e-6):
        super().__init__()
        
        # define gamma and beta
        self.gamma = nn.Parameter(torch.ones(D))
        self.beta = nn.Parameter(torch.zeros(D))

        self.eps = eps

    def forward(self, x):
        # x shape = [B, T, D], normalize within the last dimension D
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)

        return (x - mean) / (std + self.eps) * self.gamma + self.beta
        

### Test

In [5]:
# hyperparameters
D = 256
B = 32
T = 10

# input
x = torch.rand(B, T, D)

# test
ln = LayerNorm(D)
features_output = ln(x)

print("input features shape: ", x.shape)
print("output features shape: ", features_output.shape)

print("input features mean: ", x.mean(-1)[0])
print("output features mean: ", features_output.mean(-1)[0])


input features shape:  torch.Size([32, 10, 256])
output features shape:  torch.Size([32, 10, 256])
input features mean:  tensor([0.4824, 0.4832, 0.5280, 0.5018, 0.5071, 0.4899, 0.4777, 0.5209, 0.4828,
        0.4890])
output features mean:  tensor([ 2.2352e-08, -2.2352e-08,  1.1176e-08, -3.3528e-08,  9.0338e-08,
         3.3528e-08,  2.9802e-08, -1.0524e-07,  3.7253e-08,  1.0245e-07],
       grad_fn=<SelectBackward0>)
